In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm
# from scipy.interpolate import interp1d  
from scipy import stats
from matplotlib.ticker import LogLocator
from matplotlib.backends.backend_pdf import PdfPages

from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.ndimage import maximum_filter

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="EulerianReconstruction",
                                dtype='float32',codeSection = "Project_Algorithms")

#data manager class (for saving data)
DataManager_TrackedProfiles_TrackedAscentTrajectories = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING AND POST-PROCESSING FUNCTIONS
dataName='ResidenceTime' #Run This Code Only After Running "Tracked_Ascent_Trajectories" for this dataName

In [ ]:
#Loading SBF X Locations

def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence
    
def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs

def Get_SBF_xmaxs_FilePath(ModelData, DataManager):
    fileName = (
        f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def LoadOrRun_find_SBF_xmaxs(ModelData, DataManager, overwrite=False):
    filePath = Get_SBF_xmaxs_FilePath(ModelData, DataManager)
    
    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            xmaxs = pickle.load(f)
        return xmaxs

    xmaxs = find_SBF_xmaxs()

    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump(xmaxs, f)
    print(f"saved to {filePath}")
    return xmaxs

SBF_XLocations = LoadOrRun_find_SBF_xmaxs(ModelData, DataManager)

In [ ]:
#Data Loading and Post-Processing Functions
def LoadAndProcessData(subsetting_RightOfSBF=False,subsetting_time=None,dataName=dataName,
                       DataManager_TrackedProfiles=DataManager_TrackedProfiles_TrackedAscentTrajectories): #Reconstruction

    parcelTypesList = [['CL','nonCL']]; parcelTypes = parcelTypesList[0]
    trajectoriesArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'trajectoriesArray_{parcelTypes[0]}vs{parcelTypes[1]}')
    priorToAscentArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'priorToAscent_{parcelTypes[0]}vs{parcelTypes[1]}')
    SubtractMeanData(trajectoriesArrayDictionary,priorToAscentArrayDictionary,dataName)
    if dataName == 'Variables':
        SubtractSBF_XLocation(trajectoriesArrayDictionary,priorToAscentArrayDictionary,SBF_XLocations)

    if dataName == 'Entrainment':
        FixDetrainmentSign(trajectoriesArrayDictionary); FixDetrainmentSign(priorToAscentArrayDictionary)
        CalculateNetEntrainment(trajectoriesArrayDictionary); CalculateNetEntrainment(priorToAscentArrayDictionary)

    if dataName == 'ResidenceTime':
        AddTimeRelativeToInitialBLAscent(trajectoriesArrayDictionary)

    #subsetting only right of SBF
    if subsetting_RightOfSBF:
        trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
        LimitLeftOrRightOfSBF(trajectoriesArrayDictionary,trackedArrays,leftRight='right')
        LimitLeftOrRightOfSBF(priorToAscentArrayDictionary,trackedArrays,leftRight='right')
        RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    #subsetting time range
    if subsetting_time is not None:
        SubsetTime(trajectoriesArrayDictionary,priorToAscentArrayDictionary,subsetting_time)
        RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    #subsetting supersets
    for arrayDictionary in [trajectoriesArrayDictionary,priorToAscentArrayDictionary]:
        arrayDictionary=SubsetDataParcelType(arrayDictionary=arrayDictionary,
                                                         parcelType_original='CL', parcelType_subset='SBF')
        arrayDictionary=SubsetDataParcelType(arrayDictionary=arrayDictionary,
                                                         parcelType_original='CL', parcelType_subset='ColdPool')
        
    return trajectoriesArrayDictionary, priorToAscentArrayDictionary

def LimitLeftOrRightOfSBF(dictionary,trackedArrays,
                          leftRight='right'):
    leftRightIdentifier=1 if leftRight=='right' else -1
    
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            leftRightList = trackedArrays[parcelType][parcelDepth][:,4] 
            mask = leftRightList == leftRightIdentifier
            for varName in dictionary[parcelType][parcelDepth]:
                dictionary[parcelType][parcelDepth][varName]=dictionary[parcelType][parcelDepth][varName][:,mask]
    # return dictionary
    
def SubsetTime(trajectoriesArrayDictionary,priorToAscentArrayDictionary,
               subsetting_time):
    times=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    mask = (times >= (subsetting_time[0] or -np.inf)) & (times <= (subsetting_time[1] or np.inf))
    timeIndices=np.arange(ModelData.Ntime)[mask]
    
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            for varName in trajectoriesArrayDictionary[parcelType][parcelDepth]:
                data_1 = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                data_2 = priorToAscentArrayDictionary[parcelType][parcelDepth][varName]
                
                first_valid = np.argmax(~np.isnan(data_1), axis=0)
                keep_mask = np.isin(first_valid, timeIndices)
                data_1[:, ~keep_mask] = np.nan
                data_2[:, ~keep_mask] = np.nan
                
    # return trajectoriesArrayDictionary,priorToAscentArrayDictionary

def RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary):
    """
    use after applying SubsetTime()
    """
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            for varName in trajectoriesArrayDictionary[parcelType][parcelDepth]:
                data_1 = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                data_2 = priorToAscentArrayDictionary[parcelType][parcelDepth][varName]
                
                keep_mask = np.any(~np.isnan(data_1), axis=0)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = data_1[:, keep_mask]
                priorToAscentArrayDictionary[parcelType][parcelDepth][varName] = data_2[:, keep_mask]
                
    return trajectoriesArrayDictionary, priorToAscentArrayDictionary

def SubtractMeanData(trajectoriesArrayDictionary,priorToAscentArrayDictionary,dataName_Global):

    if dataName_Global != 'Variables': return
        
    DataManager_CalculateMeans = DataManager_Class(mainDirectory, scratchDirectory, ModelData, 
                                                   dataType="CalculateMoreVariables",dataName="CalculateMeans",dtype='float32',
                                                   verbose=False)
    
    [inputDataDirectory,dataName] = CallVariable(ModelData, DataManager_CalculateMeans, 'all_times', 'qv_mean',loadData=False) 
    meanData = DataManager.GetTimestepData_V2(inputDataDirectory, 'all_times', dataName=dataName)
    
    varNames = GetMeanVarNames()
    for varName in varNames:
        varNameMean = varName.lower()+'_mean'
        if varNameMean not in meanData: continue        
        varMean = meanData[varNameMean][:]  # shape (t, Nz)
        for dictionary in [trajectoriesArrayDictionary, priorToAscentArrayDictionary]:
            for parcelType in dictionary:
                for parcelDepth in dictionary[parcelType]:
                    d = dictionary[parcelType][parcelDepth]
                    zIndexes = d['Z']
                    validMask = ~np.isnan(zIndexes)
    
                    perturbation = np.full(d[varName].shape, np.nan)
                    tIndexes = np.arange(varMean.shape[0])[:, np.newaxis] * np.ones_like(zIndexes, dtype=int)
                    
                    perturbation[validMask] = d[varName][validMask] - varMean[tIndexes[validMask], zIndexes[validMask].astype(int)]    
                    dictionary[parcelType][parcelDepth][varName+'_perturbation'] = perturbation
                    
    #close data
    meanData.close()

def SubtractSBF_XLocation(trajectoriesArrayDictionary, priorToAscentArrayDictionary, SBF_XLocations):

    SBF_XLocations = np.asarray(SBF_XLocations, dtype=float)

    for dictionary in [trajectoriesArrayDictionary, priorToAscentArrayDictionary]:
        for parcelType in dictionary:
            for parcelDepth in dictionary[parcelType]:

                d = dictionary[parcelType][parcelDepth]

                xData = d['X']
                validMask = ~np.isnan(xData)

                tIndexes = np.arange(xData.shape[0])[:, np.newaxis] * np.ones_like(xData, dtype=int)

                sbfForParcels = SBF_XLocations[tIndexes]

                validMask = validMask & ~np.isnan(sbfForParcels)

                xPerturbation = np.full(xData.shape, np.nan)
                xPerturbation[validMask] = xData[validMask] - sbfForParcels[validMask]

                d['X_SBF_Relative'] = xPerturbation

def FixDetrainmentSign(dictionary):
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            for varName in dictionary[parcelType][parcelDepth]:
                PROCESSED_string = 'PROCESSED_' if 'PROCESSED_' in varName else ""
                DivideMassFlux_string = '_DivideMassFlux' if '_DivideMassFlux' in varName else ""
                if f"{PROCESSED_string}D{DivideMassFlux_string}_" in varName:
                    dictionary[parcelType][parcelDepth][varName] *= -1

def CalculateNetEntrainment(dictionary):
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            variables = dictionary[parcelType][parcelDepth]
            varNames = list(variables.keys())
            entrainment_varNames = [varName for varName in varNames if "_E_" in varName]
            for entrainment_varName in entrainment_varNames:
                detrainment_varName = entrainment_varName.replace('_E_', '_D_')
                net_varName = entrainment_varName.replace('_E_', '_NET_')
                variables[net_varName] = variables[entrainment_varName] - variables[detrainment_varName]

def AddTimeRelativeToInitialBLAscent(trajectoriesArrayDictionary):
    
    time = (np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6) * 60
    
    for parcelType in trajectoriesArrayDictionary.keys():
        for parcelDepth in trajectoriesArrayDictionary[parcelType].keys():
            dataDictionary = trajectoriesArrayDictionary[parcelType][parcelDepth]
            
            Z = dataDictionary['Z']
            z = dataDictionary['z']
            W = dataDictionary['W']
            qcqi = dataDictionary['QCQI']
            
            masks = {
                'ascent': np.isfinite(Z) & (W > 0.1),
                'CB':     np.isfinite(Z) & (qcqi > 1e-6),
                'CU':     np.isfinite(Z) & (W > 0.5) & (qcqi > 1e-6),
            }
            
            for suffix, mask in masks.items():
                first_idx = np.argmax(mask, axis=0)
                time_at_first = time[first_idx]
                
                time_since = time[:, None] - time_at_first[None, :]
                time_since = np.where(mask, time_since, np.nan)

                dataDictionary[f'z_since_{suffix}'] = np.where(mask, z, np.nan)
                dataDictionary[f'W_since_{suffix}'] = np.where(mask, W, np.nan)
                dataDictionary[f'time_since_{suffix}'] = time_since
                # AddNormalizedTimeSinceAscent(dataDictionary, key, mask)

    return trajectoriesArrayDictionary
    

# def AddNormalizedTimeSinceAscent(dataDictionary, timeKey, mask):

#     T = dataDictionary[timeKey]          # min
#     zHeight = dataDictionary['z'] / 1e3  # km

#     zMasked = np.where(mask, zHeight, np.nan)
#     zStart = np.nanmin(zMasked, axis=0)

#     dzFromStart = zMasked - zStart[None, :]

#     with np.errstate(invalid='ignore', divide='ignore'):
#         normalizedAge = T / dzFromStart

#     dataDictionary[f'{timeKey}_per_km'] = normalizedAge

def SubsetDataParcelType(arrayDictionary,parcelType_original='CL', parcelType_subset='SBF'):
    """
    In arrayDictionary, the parcelType CL is a superset of SBF and ColdPool.
    This function allows simple "isin" mask to create new parcelTypes for the subset of each superset,
    using indicies marking each subset within trackedArrays (loaded below), as calculated from the SubsetParcels code.
    """

    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class,verbose=False)
    
    original = trackedArrays[parcelType_original]['ALL']
    subset = trackedArrays[parcelType_subset]['ALL']
    
    mask = np.isin(original[:,0], subset[:,0])
    
    arrayDictionary[parcelType_subset] = {}
    
    for inner_key1 in arrayDictionary[parcelType_original]:
        arrayDictionary[parcelType_subset][inner_key1] = {}
        for inner_key2 in arrayDictionary[parcelType_original][inner_key1]:
            varData = arrayDictionary[parcelType_original][inner_key1][inner_key2]
            arrayDictionary[parcelType_subset][inner_key1][inner_key2] = varData[:, mask]
    
    return arrayDictionary

In [ ]:
#Loading Data
subsetting_RightOfSBF=False; #subsetting_RightOfSBF=True
subsetting_time=None; #subsetting_time=(12,None)

[trajectoriesArrayDictionary,priorToAscentArrayDictionary]\
=LoadAndProcessData(subsetting_RightOfSBF=subsetting_RightOfSBF,subsetting_time=subsetting_time)

In [ ]:
##############################################
#CALCULATING FUNCTIONS

In [ ]:
#Calculating Functions
def GetCoordinateData(trajectoriesArrayDictionary,parcelType='CL'):
    Z=trajectoriesArrayDictionary[parcelType]['ALL']['Z']
    Y=trajectoriesArrayDictionary[parcelType]['ALL']['Y']
    X=trajectoriesArrayDictionary[parcelType]['ALL']['X']
    return Z,Y,X
def parcelCount(Z_t, Y_t, X_t):
    """
    Z_t, Y_t, X_t: 1D arrays of indices (one entry per parcel)
    Returns:
        count_grid: (Nz, Ny, Nx) array of parcel counts per cell
    """
    #Setup
    Nz=ModelData.Nzh; Ny=ModelData.Nyh; Nx=ModelData.Nxh

    #Removing Nans
    valid = ~(np.isnan(Z_t) | np.isnan(Y_t) | np.isnan(X_t))
    Z_t_valid = Z_t[valid].astype(np.int64)
    Y_t_valid = Y_t[valid].astype(np.int64)
    X_t_valid = X_t[valid].astype(np.int64)
    
    countArray = np.zeros((Nz, Ny, Nx), dtype=np.int32)
    np.add.at(countArray, (Z_t_valid,Y_t_valid,X_t_valid), 1)
    return countArray

def GetSubsettedData(t,countArray,varName='theta_e',
                     expand_kms=1,filterType='3d'):
    var=CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName=varName)
    mask = countArray != 0

    if expand_kms is not None and expand_kms>=1: 
        expand = int(round(expand_kms * ModelData.kms))
        if filterType=='3d':
            mask = maximum_filter(mask.astype(np.uint8), size=2*expand+1) > 0 #expand in 3d
        elif filterType=='2d':
            mask = maximum_filter(mask.astype(np.uint8),size=(1, 2*expand+1, 2*expand+1)) > 0 #expand in 2d (horizontally)
        
    varArray = np.where(mask, var, np.nan)
    return varArray

def MakeCalculations(t,
                    Z,Y,X,varName='theta_e'):
    Z_t=Z[t]; Y_t=Y[t]; X_t=X[t]
    countArray = parcelCount(Z_t, Y_t, X_t)
    varArray = GetSubsettedData(t,countArray,varName=varName)
    return [countArray,varArray]

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
#Plotting Colorbar Functions
def MakeTheta_e_cmap():
    def band(c1, c2, n):
        return list(mcolors.LinearSegmentedColormap.from_list("tmp", [c1, c2], N=n)(np.linspace(0, 1, n)))

    colors = (
        band("#d9d9d9", "#666666", 11) +   # white -> gray, 11 shades
        band("#9fd89f", "#3f8f3f", 3)  +   # green, 3 shades
        band("#cfe8ff", "#7fb8e8", 5)  +   # light blue, 5 shades
        band("#3f7fc0", "#0a1f5f", 6)  +   # dark blue, 6 shades
        band("#fff84f", "#ffd23f", 4)  +   # yellow, 4 shades
        band("#ffae3f", "#ff7a1f", 3)  +   # orange, 3 shades
        band("#e8451f", "#5a0a0f", 8)      # red, 6 shades
    )

    cmap = mcolors.ListedColormap(colors, name="theta_e_cmap")

    #fixed levels
    levels = np.linspace(336, 356, cmap.N + 1) #full range about 327-374
    # levels = np.linspace(345, 365, cmap.N + 1) 
    return [cmap,levels]

def TestPlot_Theta_e_cmap():
    import matplotlib as mpl
    [cmap,_] = MakeTheta_e_cmap()
    
    fig, ax = plt.subplots(figsize=(8, 1))
    norm = mpl.colors.Normalize(vmin=336, vmax=356)
    cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal')
    cb.set_label('K')
    plt.show()

In [ ]:
#Plotting Functions (MaxParcelCount)
def PlotMaxParcelCount_YX(countArray, axis,
                          cmap=plt.cm.viridis.copy()):
    countArray_nan = countArray.astype(np.float64)
    plotData = np.nanmax(countArray_nan, axis=0)
    cmap.set_under('white')
    levels = np.arange(0, np.nanmax(plotData))
    cf = axis.contourf(ModelData.xh, ModelData.yh, plotData, levels=levels, cmap=cmap, extend='min')

    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label='max # of parcels')

    axis.set_ylim(0, 160); axis.set_xlim(100, 400)
    axis.set_ylabel('y (km)'); axis.set_xlabel('x (km)')
    
def PlotMaxParcelCount_ZY(countArray, axis,
                          cmap=plt.cm.viridis.copy()):
    countArray_nan = countArray.astype(np.float64)
    plotData = np.nanmax(countArray_nan, axis=2)
    cmap.set_under('white')
    levels = np.arange(0, np.nanmax(plotData))
    cf = axis.contourf(ModelData.yh, ModelData.zh, plotData, levels=levels, cmap=cmap, extend='min')

    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label='max # of parcels')

    axis.set_ylim(0, 16); axis.set_xlim(np.floor(ModelData.yh[0]),np.ceil(ModelData.yh[-1]))
    axis.set_ylabel('z (km)'); axis.set_xlabel('y (km)')
    
def PlotMaxParcelCount_ZX(countArray, axis,
                          cmap=plt.cm.viridis.copy()):
    countArray_nan = countArray.astype(np.float64)
    plotData = np.nanmax(countArray_nan, axis=1)
    cmap.set_under('white')
    levels = np.arange(0, np.nanmax(plotData))
    cf = axis.contourf(ModelData.xh, ModelData.zh, plotData, levels=levels, cmap=cmap, extend='min')

    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label='max # of parcels')

    axis.set_ylim(0, 16); axis.set_xlim(100, 400)
    axis.set_ylabel('z (km)'); axis.set_xlabel('x (km)')
    
def PlotMaxParcelCount_Z(countArray, axis):
    countArray_nan = countArray.astype(np.float64)
    plotData = np.nanmax(countArray_nan, axis=(1, 2))
    axis.plot(plotData, ModelData.zh)
    axis.set_xlim(0, np.nanmax(plotData))
    axis.set_ylim(0, 16)
    axis.set_ylabel('z (km)'); axis.set_xlabel('max # of parcels')

    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cax.axis('off')
    
def PlotMaxParcelCount(countArray,t=None,
                       cmap=plt.cm.turbo.copy()):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    PlotMaxParcelCount_YX(countArray, axes[0, 0], cmap=cmap)
    PlotMaxParcelCount_ZY(countArray, axes[0, 1], cmap=cmap)
    PlotMaxParcelCount_ZX(countArray, axes[1, 0], cmap=cmap)
    PlotMaxParcelCount_Z(countArray, axes[1, 1])

    if t is not None: fig.suptitle(f'Max Parcel Counts at {ModelData.time_hrs[t]:.2f} LT', fontsize=14)
    plt.tight_layout()
    return fig

In [ ]:
#Plotting Functions (Variable)    
def PlotAverageVariable_YX(variableArray, axis, label=rf'$\theta_e$ (K)',units=f'(K)',
                           plotType='pcolormesh',cmap='turbo',levels=None):
    plotData = TakeMean(variableArray,axis=0)

    if levels is None:
        vmin, vmax = np.nanmin(plotData), np.nanmax(plotData)
        levels = np.linspace(vmin, vmax, 21)
    else:
        vmin=levels[0];vmax=levels[-1]
    
    if plotType=='contourf':
        cf = axis.contourf(ModelData.xh, ModelData.yh, plotData, levels=levels, cmap=cmap)
    elif plotType=='pcolormesh':
        cf = axis.pcolormesh(ModelData.xh, ModelData.yh, plotData, cmap=cmap, vmin=vmin, vmax=vmax)
        
    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label=f"{label} {units}")
    axis.set_ylim(0, 160); axis.set_xlim(100, 400)
    axis.set_ylabel('y (km)'); axis.set_xlabel('x (km)')
    
def PlotAverageVariable_ZY(variableArray, axis, label=rf'$\theta_e$ (K)',units=f'(K)',
                           plotType='pcolormesh',cmap='turbo',levels=None):
    plotData = TakeMean(variableArray,axis=2)
    
    if levels is None:
        vmin, vmax = np.nanmin(plotData), np.nanmax(plotData)
        levels = np.linspace(vmin, vmax, 21)
    else:
        vmin=levels[0];vmax=levels[-1]
        
    if plotType=='contourf':
        cf = axis.contourf(ModelData.yh, ModelData.zh, plotData, levels=levels, cmap=cmap)
    elif plotType=='pcolormesh':
        cf = axis.pcolormesh(ModelData.yh, ModelData.zh, plotData, cmap=cmap, vmin=vmin, vmax=vmax)
        
    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label=f"{label} {units}")
    axis.set_ylim(0, 16); axis.set_xlim(np.floor(ModelData.yh[0]), np.ceil(ModelData.yh[-1]))
    axis.set_ylabel('z (km)'); axis.set_xlabel('y (km)')
    
def PlotAverageVariable_ZX(variableArray, axis, label=rf'$\theta_e$ (K)',units=f'(K)',
                           plotType='pcolormesh',cmap='turbo',levels=None):
    plotData = TakeMean(variableArray,axis=1)
    
    if levels is None:
        vmin, vmax = np.nanmin(plotData), np.nanmax(plotData)
        levels = np.linspace(vmin, vmax, 21)
    else:
        vmin=levels[0];vmax=levels[-1]
        
    if plotType=='contourf':
        cf = axis.contourf(ModelData.xh, ModelData.zh, plotData, levels=levels, cmap=cmap)
    elif plotType=='pcolormesh':
        cf = axis.pcolormesh(ModelData.xh, ModelData.zh, plotData, cmap=cmap, vmin=vmin, vmax=vmax)
        
    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label=f"{label} {units}")
    axis.set_ylim(0, 16); axis.set_xlim(100, 400)
    axis.set_ylabel('z (km)'); axis.set_xlabel('x (km)')
    
def PlotAverageVariable_Z(variableArray, axis, label=rf'$\theta_e$ (K)',units=f'(K)'):
    plotData = TakeMean(variableArray,axis=(1,2))
    axis.plot(plotData, ModelData.zh)
    
    axis.set_ylim(0, 16)
    axis.set_xlabel(f"{label} {units}"); axis.set_ylabel('z (km)')
    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cax.axis('off')
    
def PlotAverageVariable(variableArray,t=None, label=rf'$\theta_e$',units=f'(K)', plotType='pcolormesh'):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    [cmap,levels] = MakeTheta_e_cmap()
    
    PlotAverageVariable_YX(variableArray, axes[0, 0], label,units, plotType=plotType, cmap=cmap,levels=levels)
    PlotAverageVariable_ZY(variableArray, axes[0, 1], label,units, plotType=plotType, cmap=cmap,levels=levels)
    PlotAverageVariable_ZX(variableArray, axes[1, 0], label,units, plotType=plotType, cmap=cmap,levels=levels)
    PlotAverageVariable_Z(variableArray, axes[1, 1], label,units)

    if t is not None: fig.suptitle(f'Average {label} at {ModelData.time_hrs[t]:.2f} LT', fontsize=14)
    plt.tight_layout()
    return fig

def GetMaskedVariable(countArray,variableArray):
    mask = countArray.copy() != 0
    variableMasked = np.where(mask, variableArray, np.nan)
    return variableMasked

def TakeMean(var,axis):
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Mean of empty slice')
        varMean=np.nanmean(var, axis=axis)
    return varMean

In [ ]:
#Plotting Functions (CrossSections)
def Plot_CrossSections(varArray, xIndices, yIndices,
                       t=None,zlim=(0, 16),useCbarLimits=True,label=rf'$\theta_e$',units=f'(K)'):
    x, y, z = ModelData.xh, ModelData.yh, ModelData.zh

    theta_e_ZX = ExtractCrossSection(varArray, axis=1, idx_spec=yIndices)  # (z, x), collapsed over y
    theta_e_ZY = ExtractCrossSection(varArray, axis=2, idx_spec=xIndices)  # (z, y), collapsed over x

    x_loc = ResolveLocValue(x, xIndices)
    y_loc = ResolveLocValue(y, yIndices)

    # xIndices -> zooms West-East panel's x-axis; yIndices -> zooms South-North panel's x-axis (y-coordinate)
    xlim = ResolveLimFromIndices(x, xIndices) or (x.min(), x.max())
    ylim = ResolveLimFromIndices(y, yIndices) or (y.min(), y.max())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    if t is not None: fig.suptitle(f'Average {label} at {ModelData.time_hrs[t]:.2f} LT', fontsize=14)
    
    [cmap,levels] = MakeTheta_e_cmap()
    norm = mcolors.BoundaryNorm(levels, cmap.N)

    # --- West-East (Z vs X) ---
    ax = axes[0]
    plotData=theta_e_ZX
    if not useCbarLimits:
        levels = np.linspace(np.nanmin(plotData), np.nanmax(plotData), cmap.N + 1)  # cmap.N + 1 edges -> cmap.N bins
        norm = mcolors.BoundaryNorm(levels, cmap.N)
    cf = ax.contourf(x, z, plotData, levels=levels, cmap=cmap, norm=norm, extend='both')
    
    # ax.axvline(x_loc, color='black', linestyle='--', linewidth=1.5)
    ax.set_title('West-East Vertical Cross-Sectional Average')
    ax.set_xlabel('x (km)'); ax.set_ylabel('z (km)')
    ax.set_xlim(xlim)

    # --- South-North (Z vs Y) ---
    ax = axes[1]
    plotData=theta_e_ZY
    if not useCbarLimits:
        levels = np.linspace(np.nanmin(plotData), np.nanmax(plotData), cmap.N + 1)  # cmap.N + 1 edges -> cmap.N bins
        norm = mcolors.BoundaryNorm(levels, cmap.N)
    ax.contourf(y, z, plotData, levels=levels, cmap=cmap, norm=norm, extend='both')

    # ax.axvline(y_loc, color='black', linestyle='--', linewidth=1.5)
    ax.set_title('South-North Vertical Cross-Sectional Average')
    ax.set_xlabel('y (km)')
    ax.set_xlim(ylim)

    for ax in axes:
        ax.set_ylim(zlim)

    fig.subplots_adjust(bottom=0.22)
    cax = fig.add_axes([0.25, 0.08, 0.5, 0.03])
    cbar = fig.colorbar(cf, cax=cax, orientation='horizontal', ticks=levels[::2])
    cbar.set_label(f'{label} {units}')
    return fig

def ExtractCrossSection(varArray, axis, idx_spec):
    """
    varArray: 3D array (z, y, x)
    axis: which axis to collapse (1 for y, 2 for x)
    idx_spec: int (single index) or (start, stop) tuple (range -> averaged)
    """
    idx = ResolveIndexSpecification(idx_spec)
    sl = [slice(None)] * varArray.ndim
    sl[axis] = idx
    sliced = varArray[tuple(sl)]

    if isinstance(idx, slice):
        return TakeMean(sliced, axis=axis)
    else:
        return sliced  # already collapsed by direct indexing

def ResolveIndexSpecification(idx_spec):
    """Convert an int or (start, stop) tuple into something usable for indexing."""
    if isinstance(idx_spec, tuple):
        return slice(idx_spec[0], idx_spec[1])
    return idx_spec

def ResolveLocValue(coord_array, idx_spec):
    """Get the coordinate value (for the dashed-line marker) for an int or (start, stop) tuple."""
    if isinstance(idx_spec, tuple):
        return np.mean(coord_array[idx_spec[0]:idx_spec[1]])
    return coord_array[idx_spec]

def ResolveLimFromIndices(coord_array, idx_spec):
    """
    Get the (min, max) coordinate range spanned by idx_spec, for use as an axis xlim/ylim.
    If idx_spec is a single int, there's no natural range -> return None (full domain).
    """
    if isinstance(idx_spec, tuple):
        start, stop = idx_spec
        return (coord_array[start], coord_array[stop - 1])
    return None

In [ ]:
##############################################
#CALCULATING

In [ ]:
#Calculating
[Z_CL,Y_CL,X_CL]=GetCoordinateData(trajectoriesArrayDictionary,parcelType='CL')
[Z_nonCL,Y_nonCL,X_nonCL]=GetCoordinateData(trajectoriesArrayDictionary,parcelType='nonCL')

In [ ]:
##############################################
#PLOTTING

In [ ]:
#Animation
#Plotting Parcel Count
for t in tqdm(range(140,160,2)):
    [countArray_CL,varArray_CL]=MakeCalculations(t,Z_CL,Y_CL,X_CL,varName='theta_e')
    fig1 = PlotMaxParcelCount(countArray_CL,t=t)
    fig = PlotAverageVariable(varArray_CL,t=t, label=rf'$\theta_e$',units=f'(K)')
    fig = Plot_CrossSections(varArray_CL,t=t, xIndices=(150,250), yIndices=(50,150), label=rf'$\theta_e$',units=f'(K)')

#* ==> Plot and Save for All Timesteps

In [ ]:
#Plotting
for t in tqdm(range(140,142,2)):
    [countArray_nonCL,varArray_nonCL]=MakeCalculations(t,Z_nonCL,Y_nonCL,X_nonCL,varName='theta_e')
    fig1 = PlotMaxParcelCount(countArray_nonCL,t=t)
    fig = PlotAverageVariable(varArray_nonCL,t=t, label=rf'$\theta_e$',units=f'(K)')
    fig = Plot_CrossSections(varArray_nonCL,t=t, xIndices=(250,350), yIndices=(50,150), label=rf'$\theta_e$',units=f'(K)')

#* ==> Plot and Save for All Timesteps

In [ ]:
##############################################
#COMBINE INTO ANIMATION

In [ ]:
#* ==> ...